# 00 · Setup, data integrity gate & partitioning  (Phase **P0**)

This notebook is a **thin driver** — all logic lives in the versioned `fedsat` package
(`src/fedsat/`). It is designed to run top-to-bottom on **Google Colab (GPU runtime)**.

**What P0 delivers (see `PLAN.md` §3–§4 and `TODO.md`):**
1. Fetch the code + real **EuroSAT** data (no synthetic data exists anywhere — kills old bugs B1/B2).
2. **Gate G1 — integrity:** assert real data, all **10** classes present with >0 samples, correct
   total, and record a data hash. Labels come from the dataset's own `ClassLabel` (kills B3).
3. **EDA:** class distribution, sample grid, channel statistics.
4. **Partition** one real dataset into `K` *regional* clients via **Dirichlet(α)** label skew
   (we emulate "regions" — no continent data is collected), then carve a comparable global test
   set and per-client train/val/test splits.
5. **Gate G2 — disjointness:** assert every split index set is pairwise disjoint, and **save the
   partition to disk** so every regime (centralized / FL) reuses the *same* splits (kills B7/B8).

> **Before you run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.


## 1 · Get the code and check the environment

In [ ]:
import os, sys, subprocess

# --- Get the fedsat package (clone the repo) ---
REPO_URL = "https://github.com/sgogoigh/Satellite-Image-Classification-Fed-Learning.git"
REPO_DIR = "/content/Satellite-Image-Classification-Fed-Learning"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)

# --- Install deps (torch/torchvision already ship with Colab; do NOT reinstall them) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# --- Make the package importable ---
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import torch, fedsat
from fedsat.utils import get_device
print("fedsat", fedsat.__version__, "| torch", torch.__version__, "| device:", get_device())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run this cell.")


## 2 · (Recommended) Mount Google Drive for persistence

In [ ]:
# Optional but recommended: persist the HF cache, partitions, and results on Drive
# so you don't re-download / re-partition every session.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/fedsat"
except Exception as e:
    print("Drive not mounted (", e, ") -> falling back to local ./artifacts")
    DRIVE = os.path.join(REPO_DIR, "artifacts")
os.makedirs(DRIVE, exist_ok=True)
print("Artifacts dir:", DRIVE)


## 3 · Configuration

One typed `ExperimentConfig` drives everything (reused verbatim by later notebooks so comparisons
stay controlled). The four fields `dataset / num_clients / alpha / seed` define the partition file
name — **keep them identical** in notebooks 01+ so they load this same partition.


In [ ]:
from fedsat.config import ExperimentConfig
from fedsat.utils import set_seed, get_device

cfg = ExperimentConfig(
    experiment_name="p0_setup_eda",
    dataset="eurosat_rgb",
    hf_repo="blanchon/EuroSAT_RGB",
    num_clients=10,      # number of emulated regional clients
    alpha=0.5,           # Dirichlet skew: 100≈IID, 0.5 moderate, 0.1 severe non-IID
    input_size=64,       # 64 = fast (Colab-friendly); 224 = pretrained-native (+~1-2%, slower)
    seed=42,
    data_cache_dir=f"{DRIVE}/hf_cache",
    partition_dir=f"{DRIVE}/partitions",
    results_dir=f"{DRIVE}/results",
    device=get_device(),
)
set_seed(cfg.seed)
print(cfg)


## 4 · Load EuroSAT and run the **integrity gate (G1)**

Loads all 27,000 real Sentinel-2 RGB patches (10 LULC classes) from the Hugging Face Hub. The gate
**raises** if anything is off — there is deliberately no synthetic fallback in the codebase.
First run downloads ~90 MB (cached on Drive if mounted).


In [ ]:
from fedsat.data import load_eurosat, integrity_gate
from fedsat.utils import save_json

hf_ds, class_names, labels = load_eurosat(cfg.hf_repo, cache_dir=cfg.data_cache_dir)
stats = integrity_gate(class_names, labels,
                       expected_classes=cfg.num_classes, expected_total=cfg.expected_total)

print("PASSED G1 -- real data, all 10 classes present, correct total.")
print(f"total={stats['total']}  classes={stats['num_classes']}  data_hash={stats['data_hash']}")
for name, n in stats["class_counts"].items():
    print(f"  {name:<22} {n}")

save_json(os.path.join(cfg.run_dir(), "data_integrity.json"), stats)


## 5 · Exploratory data analysis

In [ ]:
import matplotlib.pyplot as plt

names = list(stats["class_counts"].keys())
vals = list(stats["class_counts"].values())
plt.figure(figsize=(11, 4))
plt.bar(names, vals, color="#2980b9")
plt.ylabel("images"); plt.title("EuroSAT class distribution (full dataset)")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()


In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for c, ax in enumerate(axes.ravel()):
    idx = int(np.where(labels == c)[0][0])          # first image of class c
    ax.imshow(hf_ds[idx]["image"].convert("RGB"))
    ax.set_title(class_names[c], fontsize=10); ax.axis("off")
plt.suptitle("EuroSAT -- one sample per class"); plt.tight_layout(); plt.show()


In [ ]:
# Per-channel RGB statistics on a random sample (context for normalization choices)
rng = np.random.default_rng(cfg.seed)
sample = rng.choice(len(labels), size=min(1000, len(labels)), replace=False)
arrs = np.stack([np.asarray(hf_ds[int(i)]["image"].convert("RGB"), dtype=np.float32) / 255.0
                 for i in sample])
print("sampled", len(sample), "images | tensor shape", arrs.shape)
print("channel means (R,G,B):", np.round(arrs.mean(axis=(0, 1, 2)), 4))
print("channel stds  (R,G,B):", np.round(arrs.std(axis=(0, 1, 2)), 4))


## 6 · Partition into regional clients + **disjointness gate (G2)**

`build_partition` (a) carves a **comparable global test set** (IID across all data, used to score
every regime on equal footing), (b) **Dirichlet(α)-partitions** the rest across `K` clients, and
(c) gives each client a stratified train/val/test split. It asserts **G2** (all index sets pairwise
disjoint) and we **save the indices to disk** so every downstream regime reuses them.


In [ ]:
from fedsat.data import build_partition, save_partition, load_partition

partition = build_partition(cfg, labels, class_names, data_hash=stats["data_hash"])
path = save_partition(cfg, partition)
print("PASSED G2 -- all split index sets are pairwise disjoint.")
print("saved partition ->", path)

# reload sanity-check (provenance)
_p = load_partition(cfg)
assert _p["meta"]["data_hash"] == stats["data_hash"], "data hash mismatch on reload!"
print("clients:", list(_p["clients"].keys()))
print("global_test size:", len(_p["global_test"]))


## 7 · Partition diagnostics (how skewed are the 'regions'?)

In [ ]:
import seaborn as sns
import pandas as pd
from fedsat.data import partition_matrix, label_entropy

mat = partition_matrix(partition, cfg.num_classes, labels)
plt.figure(figsize=(11, 5))
sns.heatmap(mat, annot=True, fmt="d", cmap="mako",
            xticklabels=[f"c{c}" for c in range(cfg.num_clients)], yticklabels=class_names)
plt.title(f"Per-client TRAIN class counts  (Dirichlet alpha={cfg.alpha})")
plt.xlabel("client"); plt.ylabel("class"); plt.tight_layout(); plt.show()

rows = []
for cid, sp in partition["clients"].items():
    rows.append({"client": cid, "n_train": len(sp["train"]), "n_val": len(sp["val"]),
                 "n_test": len(sp["test"]),
                 "label_entropy": round(label_entropy(mat[:, int(cid)]), 3)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nLower entropy = more skewed client. Max entropy for {cfg.num_classes} classes = "
      f"{round(float(np.log2(cfg.num_classes)), 3)}")


## Done — P0 complete ✅

You now have, saved on Drive:
- `data_integrity.json` (G1 passed, data hash recorded)
- a **partition file** with a comparable global test set + `K` client train/val/test splits (G2 passed)

**Next:** open **`01_centralized_baseline.ipynb`** (Phase P1) to train the centralized upper-bound
model on the *union* of client training data using this exact partition. Keep
`dataset / num_clients / alpha / seed` identical so it loads the same partition.
